# 07 — Dynamic Task Values (Parameterisation)

> **DataMindAI | Ahmed**

## Three ways to pass data between tasks

| Method | Where | Best for |
|--------|-------|---------|
| `dbutils.jobs.taskValues.set()` | Publisher notebook | Publishing values |
| `{{tasks.X.values.key}}` in UI | Job task parameter field | Consuming via widget (recommended) |
| `dbutils.jobs.taskValues.get()` | Consumer notebook | Interactive dev with `debugValue` |

## The student pipeline flow
```
Notebook X (ingest)                    Notebook Y (report)
  .set('total_records', 10)  ──────►   widgets.get('total_records')  = '10'
  .set('pass_rate', 70.0)    ──────►   widgets.get('pass_rate')      = '70.0'
  .set('at_risk', 2)         ──────►   widgets.get('at_risk')        = '2'
```

---
## Notebook X — Publisher (sets task values)
---

In [ ]:
# ── NOTEBOOK X: publisher ────────────────────────────────────────────────────
# This notebook calculates KPIs and publishes them as task values.
# Any downstream task in the same job run can read these.

from pyspark.sql import Row
from pyspark.sql.functions import avg, col, when

students = [
    Row(student_id='S001', name='Aisha Rahman',    course='Data Science', score=82, attendance=95),
    Row(student_id='S002', name='James Bradley',   course='Data Science', score=41, attendance=60),
    Row(student_id='S003', name='Priya Patel',     course='Engineering',  score=76, attendance=88),
    Row(student_id='S004', name='Carlos Mendez',   course='Engineering',  score=55, attendance=72),
    Row(student_id='S005', name='Sophie Williams', course='Mathematics',  score=91, attendance=98),
    Row(student_id='S006', name='Kwame Asante',    course='Mathematics',  score=38, attendance=55),
    Row(student_id='S007', name='Liu Wei',         course='Data Science', score=67, attendance=80),
    Row(student_id='S008', name='Emma Johnson',    course='Engineering',  score=74, attendance=85),
    Row(student_id='S009', name='Tariq Ahmed',     course='Data Science', score=29, attendance=40),
    Row(student_id='S010', name='Fatima Al-Sayed', course='Mathematics',  score=88, attendance=94),
]

df = spark.createDataFrame(students)
df.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.bronze.students_raw')

total_records = df.count()
passed        = df.filter('score >= 40').count()
at_risk       = df.filter('score < 40 OR attendance < 50').count()
avg_sc        = round(df.agg(avg('score')).collect()[0][0], 1)
pass_rate     = round((passed / total_records) * 100, 1)
top_row       = df.orderBy(col('score').desc()).first()
top_student   = f"{top_row['name']} ({top_row['score']}%)"

print('Publishing task values:')

# ── THE KEY API ──────────────────────────────────────────────────────────────
dbutils.jobs.taskValues.set(key='total_records', value=int(total_records))
dbutils.jobs.taskValues.set(key='pass_rate',     value=float(pass_rate))
dbutils.jobs.taskValues.set(key='at_risk',       value=int(at_risk))
dbutils.jobs.taskValues.set(key='avg_score',     value=float(avg_sc))
dbutils.jobs.taskValues.set(key='top_student',   value=top_student)

print(f'  total_records = {total_records}')
print(f'  pass_rate     = {pass_rate}')
print(f'  at_risk       = {at_risk}')
print(f'  avg_score     = {avg_sc}')
print(f'  top_student   = {top_student}')

---
## Notebook Y — Consumer via UI widget (recommended)
---

### Job UI Setup for Notebook Y
In the Job UI, add these parameters to Notebook Y's task:

| Parameter key | Parameter value |
|---------------|----------------|
| `total_records` | `{{tasks.notebook_x.values.total_records}}` |
| `pass_rate` | `{{tasks.notebook_x.values.pass_rate}}` |
| `at_risk` | `{{tasks.notebook_x.values.at_risk}}` |
| `avg_score` | `{{tasks.notebook_x.values.avg_score}}` |
| `top_student` | `{{tasks.notebook_x.values.top_student}}` |

In [ ]:
# ── NOTEBOOK Y: consumer via widget (recommended pattern) ────────────────────
# Receives task values via widget parameters set in the Job UI.
# The {{ }} reference is resolved by the job engine before this notebook starts.
# The notebook sees only the final string value.

try:
    total_records = int(dbutils.widgets.get('total_records'))
    pass_rate     = float(dbutils.widgets.get('pass_rate'))
    at_risk       = int(dbutils.widgets.get('at_risk'))
    avg_score     = float(dbutils.widgets.get('avg_score'))
    top_student   = dbutils.widgets.get('top_student')
except:
    # Local fallback for interactive testing
    total_records, pass_rate, at_risk = 10, 70.0, 2
    avg_score, top_student = 63.1, 'Sophie Williams (91%)'
    print('(Using local fallback values for interactive testing)')

status = 'ON TRACK' if pass_rate >= 70 else ('NEEDS ATTENTION' if pass_rate >= 50 else 'CRITICAL')

print('=' * 55)
print('   STUDENT PERFORMANCE REPORT')
print('=' * 55)
print(f'   Total students  : {total_records}')
print(f'   Pass rate       : {pass_rate}%  —  {status}')
print(f'   Average score   : {avg_score}%')
print(f'   At-risk count   : {at_risk}')
print(f'   Top performer   : {top_student}')
print('=' * 55)

---
## Notebook Z — Consumer via .get() (with debugValue)
---

In [ ]:
# ── NOTEBOOK Z: consumer via .get() with debugValue ─────────────────────────
# Use this pattern ONLY during local development.
# debugValue is used in interactive mode only — ignored in job runs.

total_records = dbutils.jobs.taskValues.get(
    taskKey    = 'notebook_x',
    key        = 'total_records',
    debugValue = 10              # fallback for interactive testing only
)

pass_rate = dbutils.jobs.taskValues.get(
    taskKey    = 'notebook_x',
    key        = 'pass_rate',
    debugValue = 70.0
)

at_risk = dbutils.jobs.taskValues.get(
    taskKey    = 'notebook_x',
    key        = 'at_risk',
    debugValue = 2
)

print('Values received via .get():')
print(f'  total_records = {total_records}')
print(f'  pass_rate     = {pass_rate}')
print(f'  at_risk       = {at_risk}')
print()
print('Note: debugValue is ONLY used in interactive mode.')
print('In a job run, the real upstream values are used instead.')

---
## Advanced — passing a dict and a list
---

In [ ]:
# ── Passing complex types: dict and list ─────────────────────────────────────
# Any JSON-serialisable type works: str, int, float, bool, list, dict
import json

# Example 1: Pass a summary dict
summary = {
    'total': 10,
    'pass_rate': 70.0,
    'at_risk': 2,
    'top_student': 'Sophie Williams'
}
dbutils.jobs.taskValues.set(key='summary', value=summary)
print('Published dict:')
print(json.dumps(summary, indent=2))
print()

# Example 2: Pass a list of at-risk student IDs
df = spark.table('demo_catalog.bronze.students_raw')
at_risk_ids = [r['student_id'] for r in df.filter('score < 40 OR attendance < 50').collect()]
dbutils.jobs.taskValues.set(key='at_risk_ids', value=at_risk_ids)
print(f'Published at_risk_ids list: {at_risk_ids}')
print()

# Example 3: Pass a full list of dicts (for For-Each input)
at_risk_details = [
    {'student_id': r['student_id'], 'name': r['name'], 'score': r['score']}
    for r in df.filter('score < 40 OR attendance < 50').collect()
]
dbutils.jobs.taskValues.set(key='at_risk_list', value=at_risk_details)
print(f'Published at_risk_list ({len(at_risk_details)} entries):')
print(json.dumps(at_risk_details, indent=2))